# Embedding behaviour test

Tiny, self-contained notebook to reproduce what we found about the AIID incident vectors
(the ones stored in `incidents.embedding.vector`, produced by the Longformer model).

It shows two things:
1. **Raw** cosine similarity sits at ~0.99 for *every* pair, related or not (anisotropy).
2. **Mean-centering** the vectors first spreads the scores out so topics actually separate.

Run it with the repo `.venv` (needs only `numpy` + `bson`, both already installed).
Point `DUMP` at a mongodump `aiidprod` dir that has `incidents.bson`.

In [1]:
import os
import bson                      # ships with pymongo
import numpy as np
from pathlib import Path

# Portable: walk up from the working directory to find the dump.
# Override with the AIID_DUMP environment variable if yours lives elsewhere.
REL = "data/snapshots/backup-20260629114136/mongodump_full_snapshot/aiidprod"
DUMP = Path(os.environ["AIID_DUMP"]) if "AIID_DUMP" in os.environ else next(
    (p / REL for p in [Path.cwd(), *Path.cwd().parents] if (p / REL).exists()), Path(REL)
)
assert DUMP.exists(), f"dump not found at {DUMP} - set AIID_DUMP to its path"

def load_incident_vectors(dump_dir):
    """Read incidents.bson -> (ids, {id: title}, matrix M of shape [n, dim])."""
    with open(Path(dump_dir) / "incidents.bson", "rb") as f:
        docs = list(bson.decode_file_iter(f))
    ids, titles, vecs = [], {}, []
    for d in docs:
        emb = d.get("embedding")
        if isinstance(emb, dict) and isinstance(emb.get("vector"), list):
            iid = int(d["incident_id"])
            ids.append(iid)
            titles[iid] = d.get("title", "")
            vecs.append(emb["vector"])
    return ids, titles, np.asarray(vecs, dtype=float)

ids, titles, M = load_incident_vectors(DUMP)
index = {iid: k for k, iid in enumerate(ids)}   # incident_id -> row in M
MEAN  = M.mean(axis=0)                           # the shared "everyone points here" direction
print(f"loaded {len(ids)} incidents, vector dim = {M.shape[1]}")

loaded 1548 incidents, vector dim = 768


## 1. Compare one health incident to related and unrelated incidents

`raw` = plain cosine on the stored vectors. `centered` = cosine after subtracting `MEAN`.
Watch the raw column stay glued near 0.99 while the centered column pulls apart.

In [2]:
def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

# Seed incidents to test with (change these to try others).
SEPSIS   = 123     # public health  - hospital sepsis prediction algorithm
OPTUM    = 124     # public health  - biased health risk scores
CRUISE   = 293     # unrelated      - self-driving car crash
DEEPFAKE = 1214    # unrelated      - deepfake political video

def compare(a, b):
    raw = cosine(M[index[a]],        M[index[b]])
    cen = cosine(M[index[a]] - MEAN, M[index[b]] - MEAN)
    print(f"#{a} vs #{b:<5}  raw={raw:.4f}   centered={cen:+.4f}   {titles[b][:45]}")

print(f"anchor #{SEPSIS}: {titles[SEPSIS]}\n")
compare(SEPSIS, OPTUM)      # health vs health        -> centered HIGH
compare(SEPSIS, CRUISE)     # health vs self-driving  -> centered low
compare(SEPSIS, DEEPFAKE)   # health vs deepfake      -> centered negative

anchor #123: Epic Systems’s Sepsis Prediction Algorithms Revealed to Have High Error Rates on Seriously Ill Patients

#123 vs #124    raw=0.9994   centered=+0.7864   Optum Algorithmic Health Risk Scores Reported
#123 vs #293    raw=0.9981   centered=+0.1398   Cruise’s Self-Driving Car Involved in a Multi
#123 vs #1214   raw=0.9925   centered=-0.2004   Donald Trump Reportedly Posts Purported AI-Mo


## 2. Nearest neighbours: raw ranking vs centered ranking

Both find #124 first, so the tiny raw differences do carry some order. But the centered
list is cleaner (more health incidents, fewer loosely-related ones).

In [4]:
def nearest(seed_id, center=False, k=10):
    base = (M - MEAN) if center else M
    unit = base / np.linalg.norm(base, axis=1, keepdims=True)
    sims = unit @ unit[index[seed_id]]
    ranked = sorted(zip(sims.tolist(), ids), reverse=True)
    return [(s, i, titles[i]) for s, i in ranked if i != seed_id][:k]

for center in (False, True):
    print(f"{'CENTERED' if center else 'RAW'} nearest to #{SEPSIS}:")
    for s, i, t in nearest(SEPSIS, center=center):
        print(f"  {s:+.4f}  #{i:<5} {t[:52]}")
    print()

RAW nearest to #123:
  +0.9994  #124   Optum Algorithmic Health Risk Scores Reportedly Unde
  +0.9990  #5     Collection of Robotic Surgery Malfunctions
  +0.9989  #339   Students Reportedly Used Generative Text Tools to Co
  +0.9989  #40    COMPAS Algorithm Reportedly Performs Poorly in Crime
  +0.9988  #561   OpenAI Alleged by Lawsuit Violated Users' Privacy Ri
  +0.9988  #138   University of Illinois' Proctorio Remote-Proctoring 
  +0.9987  #11    Northpointe Risk Models
  +0.9987  #498   GPT-4 Reportedly Posed as Blind Person to Convince H
  +0.9986  #54    Predictive Policing Biases of PredPol
  +0.9986  #79    Kidney Testing Method Allegedly Underestimated Risk 

CENTERED nearest to #123:
  +0.7864  #124   Optum Algorithmic Health Risk Scores Reportedly Unde
  +0.6310  #5     Collection of Robotic Surgery Malfunctions
  +0.5931  #79    Kidney Testing Method Allegedly Underestimated Risk 
  +0.5753  #81    Researchers find evidence of racial, gender, and soc
  +0.5604  #102   Pers

## 3. How compressed is the raw signal?

The raw scores to an anchor live in a band only ~0.02 wide (tiny std). Centering stretches
the *same* signal about 100x. That width, not floating-point precision, is why raw thresholds
are fragile.

In [5]:
def sims_to(anchor, center=False):
    base = (M - MEAN) if center else M
    unit = base / np.linalg.norm(base, axis=1, keepdims=True)
    return unit @ unit[index[anchor]]

ids_arr = np.array(ids)
for center in (False, True):
    s = sims_to(SEPSIS, center)[ids_arr != SEPSIS]     # exclude the anchor itself
    label = "centered" if center else "raw     "
    print(f"{label}: min={s.min():+.4f}  max={s.max():+.4f}  std={s.std():.4f}")

raw     : min=+0.9810  max=+0.9994  std=0.0019
centered: min=-0.4596  max=+0.7864  std=0.1914


## Try your own

`compare(a, b)` for any two incident ids, and `nearest(seed, center=True)` for any seed.
Example below uses two self-driving incidents.

In [6]:
compare(CRUISE, 353)                       # two self-driving incidents
for s, i, t in nearest(CRUISE, center=True, k=5):
    print(f"  {s:+.4f}  #{i:<5} {t[:52]}")

#293 vs #353    raw=0.9980   centered=+0.5707   Tesla on Autopilot Crashed into Trailer Truck
  +0.7197  #181   BMW Sedan Made a Prohibited Left Turn, Colliding wit
  +0.6907  #321   Tesla Model X on Autopilot Crashed into California H
  +0.6750  #151   California Regulator Suspended Pony.ai's Driverless 
  +0.6333  #434   Sudden Braking by Tesla Allegedly on Self-Driving Mo
  +0.6320  #333   Tesla on Autopilot Crashed Parked Michigan Police Ca
